In [0]:
# =============================================================================
# Streaming Table: fact_orders (Streaming Query with Join)
# =============================================================================
from pyspark.pipelines import materialized_view, table
from pyspark.sql import functions as F

@table(
    comment="Enriched orders joining streaming orders with customer dimensions",
    table_properties={
        "delta.autoOptimize.optimizeWrite": "true",
        "delta.autoOptimize.autoCompact": "true"
    }
)
def fact_orders():
    """
    Streaming fact table joining orders with customer dimensions.

    Sources:
        - raw_orders (streaming): Real-time order events
        - customers (batch): Customer dimension data

    Join Pattern: Stream-static join
        - Streaming side: raw_orders
        - Static side: customers (materialized view)

    Runs: Continuously (follows raw_orders stream)

    Schema:
        order_id: string
        customer_id: int
        customer_name: string (from customers)
        email: string (from customers)
        region: string (from customers)
        product_id: string
        quantity: int
        amount: decimal(10,2)
        order_date: timestamp

    Business Logic:
        - INNER JOIN: Only include orders with valid customers
        - Enriches orders with customer attributes for analytics
        - Enables regional analysis, customer segmentation, etc.

    Performance Notes:
        - Stream-static joins are efficient (static side cached)
        - Auto-optimize enabled for write performance
        - Delta format provides ACID guarantees
    """
    # Read streaming orders
    orders_stream = spark.readStream.table("raw_orders")

    # Read static customers (materialized view)
    customers_df = spark.read.table("customers")

    # Perform stream-static join
    return (orders_stream
            .join(customers_df, on="customer_id", how="inner")
            .select(
                "order_id",
                "customer_id",
                F.col("name").alias("customer_name"),
                "email",
                "region",
                "product_id",
                "quantity",
                "amount",
                "order_date"
            ))